# Training memory and runtime

Memory budgets, activations, and estimates from measured steps.

> All examples run on CPU without model downloads and use synthetic data. They explain mechanisms; real training quality must be evaluated separately.

## What to understand

Training memory includes weights, gradients, optimizer states, activations, and temporary buffers. FP32 AdamW commonly needs about 16 bytes per trainable parameter for weights, gradients, and moments alone; mixed precision and fused optimizers change this total. Attention and context length can dominate peak memory.

We compute a lower-bound FP32 estimate and time CPU steps. This does not predict GPU throughput or guarantee fit. Use preflight and a short benchmark on the target hardware with matching batch, resolution, precision, and optimizer. Startup includes warmup, data loading, and compilation.

In [ ]:
%matplotlib inline
import torch
from torch import nn
from torch.nn import functional as F
import matplotlib.pyplot as plt
torch.set_num_threads(1)
torch.manual_seed(42)
plt.rcParams.update({'figure.figsize': (7, 3), 'axes.spines.top': False, 'axes.spines.right': False})
print('PyTorch:', torch.__version__, '| device: CPU')

## Run the experiment

Run cells in order. Edit the code; Shift + Enter runs the current cell.

In [ ]:
import time, statistics
model = nn.Sequential(nn.Linear(64, 128), nn.GELU(), nn.Linear(128, 16))
optimizer = torch.optim.AdamW(model.parameters())
x = torch.randn(16, 64); target = torch.randn(16, 16)
count = sum(p.numel() for p in model.parameters())
print('FP32 weights + gradients + Adam moments, MiB:', count * 16 / 2**20)
durations = []
for step in range(15):
    begin = time.perf_counter()
    optimizer.zero_grad(set_to_none=True); loss = F.mse_loss(model(x), target)
    loss.backward(); optimizer.step()
    if step >= 5:
        durations.append(time.perf_counter() - begin)
median = statistics.median(durations)
print('measured CPU step, ms:', median * 1000)
print('1000 steps, compute only, seconds:', median * 1000)

## Try it yourself

Increase batch size fourfold and remeasure. Why might runtime scale differently? Add activation and checkpoint-loading headroom to the memory budget.

Save a notebook copy and record what you changed, expected, and measured.

## Continue in Studio

The workspace link above the lesson opens related tools. This notebook does not launch large training jobs. Before real training, check data, resources, and method support for the chosen model.

## Sources

- [PyTorch CUDA memory](https://docs.pytorch.org/docs/stable/notes/cuda.html#memory-management)